# Assignment 5 — Comparing RNN, LSTM and GRU

**Platform:** Google Colab &nbsp;|&nbsp; **Suggested runtime:** GPU  
**How to use:** Run the cells from top to bottom. Change the small experiment
constants when more training time is available.

This workbook is written as a compact college assignment: it explains the
problem, implements the method, evaluates the result, and records the main
observations.


## Problem and experiment

Classify IMDB movie reviews as positive or negative. SimpleRNN, LSTM,
and GRU models receive the same tokenized data, embedding size, recurrent
units, optimizer, and epoch limit. This controls most variables so the
recurrent layer is the main difference.

**Metrics:** accuracy, precision, recall, F1 score, parameter count, and
training time. The default subset keeps the Colab exercise practical.


In [ ]:
import gc
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

VOCAB_SIZE = 10_000
MAX_LENGTH = 200
TRAIN_SAMPLES = 12_000   # increase for a stronger final model
TEST_SAMPLES = 4_000

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE, seed=SEED
)
x_train = tf.keras.utils.pad_sequences(x_train[:TRAIN_SAMPLES], maxlen=MAX_LENGTH)
y_train = y_train[:TRAIN_SAMPLES]
x_test = tf.keras.utils.pad_sequences(x_test[:TEST_SAMPLES], maxlen=MAX_LENGTH)
y_test = y_test[:TEST_SAMPLES]
print(x_train.shape, x_test.shape)


In [ ]:
def build_sequence_model(layer_name):
    layer_class = {
        "SimpleRNN": tf.keras.layers.SimpleRNN,
        "LSTM": tf.keras.layers.LSTM,
        "GRU": tf.keras.layers.GRU,
    }[layer_name]
    model = tf.keras.Sequential([
        tf.keras.layers.Input((MAX_LENGTH,)),
        tf.keras.layers.Embedding(VOCAB_SIZE, 64, mask_zero=True),
        layer_class(32),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ], name=layer_name.lower())
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

results, histories = [], {}
for name in ["SimpleRNN", "LSTM", "GRU"]:
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(SEED)
    model = build_sequence_model(name)
    start = time.perf_counter()
    history = model.fit(
        x_train, y_train, validation_split=0.20, epochs=4,
        batch_size=128, verbose=1,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=1, restore_best_weights=True
        )],
    )
    seconds = time.perf_counter() - start
    prediction = (model.predict(x_test, batch_size=256, verbose=0).ravel() >= 0.5)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, prediction, average="binary", zero_division=0
    )
    results.append({
        "model": name, "accuracy": accuracy_score(y_test, prediction),
        "precision": precision, "recall": recall, "f1": f1,
        "parameters": model.count_params(), "train_seconds": seconds,
    })
    histories[name] = history.history

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df.round(3))


In [ ]:
results_df.set_index("model")[["accuracy", "precision", "recall", "f1"]].plot(
    kind="bar", ylim=(0.5, 1.0), figsize=(10, 4)
)
plt.ylabel("Score"); plt.title("Sequence-classification metrics")
plt.xticks(rotation=0); plt.legend(loc="lower right"); plt.show()

plt.figure(figsize=(8, 4))
for name, hist in histories.items():
    plt.plot(hist["val_accuracy"], marker="o", label=name)
plt.xlabel("Epoch"); plt.ylabel("Validation accuracy")
plt.title("Validation learning curves"); plt.legend(); plt.show()


## Discussion

SimpleRNN has the simplest state update but can struggle with long-term
dependencies. LSTM uses input, forget, and output gates. GRU uses fewer
gates and often trains faster. The best architecture is the one that gives
a strong F1 score with acceptable time and parameter cost on this run.


## Conclusion

The experiment above provides a complete training and evaluation workflow. The
printed metrics and plots are the result for the current run and should be used
to identify the strongest behaviour, the main limitation, and one justified
improvement. Exact values may vary slightly because neural-network training is
stochastic.
